# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'
# Load the dataset
dataset = mlc.Dataset(croissant_url)
# Access dataset metadata
metadata_obj = dataset.metadata
print(f"Name: {getattr(metadata_obj, 'name', None)}\nDescription: {getattr(metadata_obj, 'description', None)}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We'll list all available record set `@id`s and show their associated fields and columns. All references use the Croissant `@id` for consistency.

In [ ]:
# List all available record sets and their fields/columns
record_sets_info = []
for rs in getattr(metadata_obj, 'record_sets', []):
    info = {'@id': rs.id, 'name': getattr(rs, 'name', None)}
    fields = []
    if getattr(rs, 'fields', None):
        for f in rs.fields:
            fields.append({'@id': f.id, 'name': getattr(f, 'name', None)})
    info['fields'] = fields
    columns = []
    if getattr(rs, 'columns', None):
        for c in rs.columns:
            columns.append({'@id': c.id, 'name': getattr(c, 'name', None)})
    info['columns'] = columns
    record_sets_info.append(info)

for i, rs in enumerate(record_sets_info):
    print(f"Record set {i+1}: @id='{rs['@id']}' Name='{rs['name']}'")
    print("  Fields:")
    for f in rs['fields']:
        print(f"    - @id='{f['@id']}' Name='{f['name']}'")
    if rs['columns']:
        print("  Columns:")
        for c in rs['columns']:
            print(f"    - @id='{c['@id']}' Name='{c['name']}'")
    print("-")

# If there are no record sets, inform the user (should not happen for proper Croissant datasets)
if not record_sets_info:
    print("No record sets found in the schema.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.
We'll pick the first listed record set (if available). Reference all entities by their `@id`.

For this dataset, we'll programmatically select a record set.

In [ ]:
# Extract data from all available record sets
dataframes = {}
record_set_ids = [rs['@id'] for rs in record_sets_info]

for record_set_id in record_set_ids:
    print(f"Loading records from record set @id='{record_set_id}'...")
    try:
        records_iter = dataset.records(record_set=record_set_id)
        df = pd.DataFrame(list(records_iter))
        if not df.empty:
            dataframes[record_set_id] = df
            print(f"Columns: {df.columns.tolist()}")
            print(df.head())
        else:
            print(f"No records found for record set {record_set_id}.")
    except Exception as e:
        print(f"Error loading records for record set '{record_set_id}':", e)

# Choose the first record set for downstream examples
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    df_main = dataframes.get(main_record_set_id, pd.DataFrame())
    print(f"Using record set @id='{main_record_set_id}' for EDA.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section demonstrates how to dynamically handle field selection by `@id`.


In [ ]:
import numpy as np

# Find a numeric field in the main record set, prioritizing common numeric names if possible
numeric_field = None
if not df_main.empty:
    for c in df_main.columns:
        if pd.api.types.is_numeric_dtype(df_main[c]):
            numeric_field = c
            break
    # If heuristic fails, try to find probable numeric field names
    if numeric_field is None:
        for name in df_main.columns:
            if any(x in name.lower() for x in ["abundance", "count", "coverage", "score", "weight", "mw"]):
                numeric_field = name
                break

if numeric_field is not None:
    print(f"Selected numeric field for analysis: '{numeric_field}'")
    # Choose a threshold as the 75th percentile as an example
    try:
        threshold = df_main[numeric_field].quantile(0.75)
    except Exception:
        threshold = 10
    filtered_df = df_main[df_main[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df = filtered_df.copy()
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a non-numeric field
    group_field = None
    for col in df_main.columns:
        if (col != numeric_field) and pd.api.types.is_string_dtype(df_main[col]):
            group_field = col
            break
    if group_field is not None:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped data by '{group_field}':")
        print(grouped_df.head())
    else:
        print("No suitable group field (categorical) was found for grouping.")
else:
    print("No numeric field found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using matplotlib or seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot a histogram of the numeric field, if available
if not df_main.empty and numeric_field is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(df_main[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of '{numeric_field}'")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    if group_field is not None:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field, y=numeric_field, data=df_main)
        plt.title(f"'{numeric_field}' by '{group_field}'")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
We have demonstrated step-by-step data access, analysis, and visualization of the [Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library. This workflow enables programmatic FAIR exploration and reproducible research based on the Croissant schema and accretive data processing using `@id` referencing for all entities.

- Data was loaded directly from the Croissant JSON-LD schema
- Schema-driven referencing of record sets, fields, and columns was demonstrated
- Exploratory steps and visualizations facilitated numeric and categorical analysis

Continue exploring using the variable and @id driven workflow on additional record sets or with other Croissant-compatible tools!